# Step 3: Shuffle & Partitioning

## Learning Objectives
1. What Shuffle is and why it is expensive
2. Identifying operations that cause Shuffle
3. Partition concepts and the effect of partition count
4. repartition vs coalesce
5. Minimizing Shuffle through partitioning strategies
6. Monitoring Shuffle in the Spark UI

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
import time

spark = SparkSession.builder \
    .appName("Step3-Shuffle-Partitioning") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Default shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Default parallelism: {sc.defaultParallelism}")
print(f"✅ Spark UI: http://localhost:4040")

Default shuffle partitions: 200
Default parallelism: 2
✅ Spark UI: http://localhost:4040


---
## 1. What is Shuffle?

**Shuffle** = Moving data between nodes to redistribute it by key

```
Before Shuffle (random distribution across partitions)
┌──────────┐  ┌──────────┐  ┌──────────┐
│ P0       │  │ P1       │  │ P2       │
│ (A,1)    │  │ (A,3)    │  │ (B,2)    │
│ (B,4)    │  │ (C,1)    │  │ (A,5)    │
│ (C,2)    │  │ (B,3)    │  │ (C,4)    │
└──────────┘  └──────────┘  └──────────┘
        │           │           │
        └───── Shuffle ─────────┘  ← network + disk I/O
                    │
┌──────────┐  ┌──────────┐  ┌──────────┐
│ P0 (A)   │  │ P1 (B)   │  │ P2 (C)   │
│ (A,1)    │  │ (B,4)    │  │ (C,2)    │
│ (A,3)    │  │ (B,2)    │  │ (C,1)    │
│ (A,5)    │  │ (B,3)    │  │ (C,4)    │
└──────────┘  └──────────┘  └──────────┘
After Shuffle (same key is in same partition)
```

### Why Shuffle is Expensive
1. **Disk I/O** — writes intermediate results to local disk
2. **Network I/O** — transfers data between nodes
3. **Serialization/Deserialization** — data encoding/decoding cost
4. **Memory pressure** — memory used for buffering

---
## 2. Identifying Operations That Cause Shuffle

In [2]:
random.seed(42)
n = 1_000_000
departments = ["Engineering", "Marketing", "Sales", "HR", "Finance"]

data = [
    (i, f"emp_{i}", random.choice(departments), random.randint(50000, 150000))
    for i in range(n)
]
df = spark.createDataFrame(data, ["id", "name", "department", "salary"])
df.cache()
df.count()  # warm up cache

print(f"Row count: {df.count():,}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Row count: 1,000,000
Partitions: 4


In [3]:
# Operations that do NOT trigger Shuffle (Narrow Transformation)
print("=== Narrow Transformation (No Shuffle) ===")
print("\n--- map / filter / select ---")
df.filter(F.col("salary") > 100000).select("name", "salary").explain()

print("\n💡 No Exchange node → No Shuffle")

=== Narrow Transformation (No Shuffle) ===

--- map / filter / select ---
== Physical Plan ==
*(1) Filter (isnotnull(salary#3L) AND (salary#3L > 100000))
+- InMemoryTableScan [name#1, salary#3L], [isnotnull(salary#3L), (salary#3L > 100000)]
      +- InMemoryRelation [id#0L, name#1, department#2, salary#3L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Scan ExistingRDD[id#0L,name#1,department#2,salary#3L]



💡 No Exchange node → No Shuffle


In [4]:
# Operations that DO trigger Shuffle (Wide Transformation)
print("=== Wide Transformation (Shuffle occurs) ===")

print("\n--- 1. groupBy + agg ---")
df.groupBy("department").agg(F.avg("salary")).explain()

print("\n--- 2. orderBy (global sort) ---")
df.orderBy("salary").explain()

print("\n--- 3. distinct ---")
df.select("department").distinct().explain()

print("\n--- 4. join ---")
dept_df = spark.createDataFrame(
    [(d, f"{d} Division") for d in departments],
    ["department", "division"]
)
df.join(dept_df, "department").explain()

print("\n💡 If you see Exchange or hashpartitioning → Shuffle is occurring")

=== Wide Transformation (Shuffle occurs) ===

--- 1. groupBy + agg ---
== Physical Plan ==
*(2) HashAggregate(keys=[department#2], functions=[avg(salary#3L)])
+- Exchange hashpartitioning(department#2, 200), ENSURE_REQUIREMENTS, [plan_id=86]
   +- *(1) HashAggregate(keys=[department#2], functions=[partial_avg(salary#3L)])
      +- InMemoryTableScan [department#2, salary#3L]
            +- InMemoryRelation [id#0L, name#1, department#2, salary#3L], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- *(1) Scan ExistingRDD[id#0L,name#1,department#2,salary#3L]



--- 2. orderBy (global sort) ---
== Physical Plan ==
*(1) Sort [salary#3L ASC NULLS FIRST], true, 0
+- Exchange rangepartitioning(salary#3L ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=106]
   +- InMemoryTableScan [id#0L, name#1, department#2, salary#3L]
         +- InMemoryRelation [id#0L, name#1, department#2, salary#3L], StorageLevel(disk, memory, deserialized, 1 replicas)
               +- *(1) Sca

In [5]:
# Summary of Shuffle-triggering operations
print("""
┌────────────────────────────────────────────────┐
│         Shuffle Summary                        │
├────────────────────┬───────────────────────────┤
│ No Shuffle         │ Shuffle occurs            │
│ (Narrow)           │ (Wide)                    │
├────────────────────┼───────────────────────────┤
│ map / select       │ groupBy + agg             │
│ filter / where     │ reduceByKey               │
│ withColumn         │ join (most cases)         │
│ union              │ distinct / dropDuplicates │
│ coalesce (shrink)  │ orderBy / sort            │
│ broadcast join     │ repartition               │
│                    │ window functions          │
└────────────────────┴───────────────────────────┘
""")


┌────────────────────────────────────────────────┐
│         Shuffle Summary                        │
├────────────────────┬───────────────────────────┤
│ No Shuffle         │ Shuffle occurs            │
│ (Narrow)           │ (Wide)                    │
├────────────────────┼───────────────────────────┤
│ map / select       │ groupBy + agg             │
│ filter / where     │ reduceByKey               │
│ withColumn         │ join (most cases)         │
│ union              │ distinct / dropDuplicates │
│ coalesce (shrink)  │ orderBy / sort            │
│ broadcast join     │ repartition               │
│                    │ window functions          │
└────────────────────┴───────────────────────────┘



---
## 3. Understanding Partitions

**Partition** = logical unit of data division. One task processes one partition.

- **Too few partitions** → insufficient parallelism, risk of OOM
- **Too many partitions** → task overhead, small file problem
- **Rule of thumb**: ~100MB–200MB per partition

In [6]:
# Utility to inspect data distribution per partition
def show_partition_info(df, label=""):
    part_sizes = df.withColumn("partition_id", F.spark_partition_id()) \
        .groupBy("partition_id") \
        .count() \
        .orderBy("partition_id") \
        .collect()
    
    counts = [row["count"] for row in part_sizes]
    print(f"\n{'=' * 50}")
    print(f"{label}")
    print(f"{'=' * 50}")
    print(f"Partitions: {len(counts)}")
    print(f"Total rows: {sum(counts):,}")
    print(f"Rows/partition — min: {min(counts):,}, max: {max(counts):,}, avg: {sum(counts)//len(counts):,}")
    
    skew_ratio = max(counts) / max(min(counts), 1)
    print(f"Skew ratio (max/min): {skew_ratio:.1f}x")
    if skew_ratio > 3:
        print("⚠️  Severe data skew detected!")
    else:
        print("✅ Partition distribution is balanced.")

show_partition_info(df, "Original DataFrame")


Original DataFrame
Partitions: 4
Total rows: 1,000,000
Rows/partition — min: 249,856, max: 250,432, avg: 250,000
Skew ratio (max/min): 1.0x
✅ Partition distribution is balanced.


In [7]:
# Performance impact of different partition counts
results = []

for num_partitions in [2, 8, 50, 200, 1000]:
    spark.conf.set("spark.sql.shuffle.partitions", str(num_partitions))
    
    start = time.time()
    df.groupBy("department").agg(
        F.count("*"),
        F.avg("salary"),
        F.max("salary")
    ).collect()
    elapsed = time.time() - start
    
    results.append((num_partitions, elapsed))
    print(f"Partitions {num_partitions:>5} → {elapsed:.3f}s")

print("\n💡 Too few or too many partitions both hurt performance.")
print("   For this data size, a middle value is optimal.")

# Restore default
spark.conf.set("spark.sql.shuffle.partitions", "200")

Partitions     2 → 0.264s
Partitions     8 → 0.134s
Partitions    50 → 0.188s
Partitions   200 → 0.357s
Partitions  1000 → 1.314s

💡 Too few or too many partitions both hurt performance.
   For this data size, a middle value is optimal.


---
## 4. repartition vs coalesce

| | repartition | coalesce |
|---|---|---|
| **Direction** | Increase or decrease | Decrease only |
| **Shuffle** | Always (Full Shuffle) | None |
| **Even distribution** | Guaranteed | Not guaranteed (merges existing partitions) |
| **Use case** | Key-based partitioning, increasing partition count | Reducing partition count before file output |

In [8]:
print(f"Original partition count: {df.rdd.getNumPartitions()}")

# repartition: causes Full Shuffle
start = time.time()
repartitioned = df.repartition(10)
repartitioned.count()  # trigger with action
repart_time = time.time() - start

print(f"\nrepartition(10): {repart_time:.3f}s")
show_partition_info(repartitioned, "repartition(10) result")

# coalesce: merges partitions without Shuffle
start = time.time()
coalesced = df.coalesce(10)
coalesced.count()
coalesce_time = time.time() - start

print(f"\ncoalesce(10): {coalesce_time:.3f}s")
show_partition_info(coalesced, "coalesce(10) result")

Original partition count: 4

repartition(10): 0.184s

repartition(10) result
Partitions: 10
Total rows: 1,000,000
Rows/partition — min: 99,999, max: 100,002, avg: 100,000
Skew ratio (max/min): 1.0x
✅ Partition distribution is balanced.

coalesce(10): 0.069s

coalesce(10) result
Partitions: 4
Total rows: 1,000,000
Rows/partition — min: 249,856, max: 250,432, avg: 250,000
Skew ratio (max/min): 1.0x
✅ Partition distribution is balanced.


In [9]:
# Compare execution plans
print("=== repartition Execution Plan ===")
df.repartition(10).explain()

print("\n=== coalesce Execution Plan ===")
df.coalesce(10).explain()

print("\n💡 repartition → Exchange (Shuffle) present")
print("   coalesce → No Exchange (partition merge only)")

=== repartition Execution Plan ===
== Physical Plan ==
Exchange RoundRobinPartitioning(10), REPARTITION_BY_NUM, [plan_id=578]
+- InMemoryTableScan [id#0L, name#1, department#2, salary#3L]
      +- InMemoryRelation [id#0L, name#1, department#2, salary#3L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Scan ExistingRDD[id#0L,name#1,department#2,salary#3L]



=== coalesce Execution Plan ===
== Physical Plan ==
Coalesce 10
+- InMemoryTableScan [id#0L, name#1, department#2, salary#3L]
      +- InMemoryRelation [id#0L, name#1, department#2, salary#3L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Scan ExistingRDD[id#0L,name#1,department#2,salary#3L]



💡 repartition → Exchange (Shuffle) present
   coalesce → No Exchange (partition merge only)


In [10]:
# Key-based repartition
# Rows with the same department value are co-located in the same partition
keyed = df.repartition(5, "department")

print("=== Key-based repartition(5, 'department') ===")
keyed.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id", "department") \
    .count() \
    .orderBy("partition_id", "department") \
    .show(30)

print("💡 Rows with the same department are in the same partition_id.")
print("   Subsequent groupBy/join on department can avoid Shuffle.")

=== Key-based repartition(5, 'department') ===
+------------+-----------+------+
|partition_id| department| count|
+------------+-----------+------+
|           1|         HR|199410|
|           1|  Marketing|199228|
|           2|Engineering|200620|
|           2|    Finance|200009|
|           2|      Sales|200733|
+------------+-----------+------+

💡 Rows with the same department are in the same partition_id.
   Subsequent groupBy/join on department can avoid Shuffle.


---
## 5. Shuffle Minimization Strategies

### 5.1 Eliminating Shuffle via Pre-Partitioning

In [11]:
# Scenario: run groupBy on department multiple times
spark.conf.set("spark.sql.shuffle.partitions", "5")

# Strategy 1: Shuffle every time
start = time.time()
df.groupBy("department").agg(F.avg("salary")).collect()
df.groupBy("department").agg(F.max("salary")).collect()
df.groupBy("department").agg(F.count("*")).collect()
no_prepart_time = time.time() - start

# Strategy 2: Pre-partition + cache
start = time.time()
pre_partitioned = df.repartition(5, "department").cache()
pre_partitioned.count()  # warm up cache (1 Shuffle)

pre_partitioned.groupBy("department").agg(F.avg("salary")).collect()
pre_partitioned.groupBy("department").agg(F.max("salary")).collect()
pre_partitioned.groupBy("department").agg(F.count("*")).collect()
prepart_time = time.time() - start

print(f"Shuffle every time:      {no_prepart_time:.3f}s")
print(f"Pre-partition + cache:   {prepart_time:.3f}s")
print("""
⚠️  Here pre-partitioning is usually NOT faster — and that itself is the lesson.
   These aggregations use combinable functions (avg / max / count), so each groupBy already
   does map-side partial aggregation (look for 'partial_avg' in the plan) and shuffles only a
   handful of rows per partition. The one-time repartition + cache adds a full shuffle that
   those already-cheap aggregations never get to amortize.

   Pre-partitioning (or bucketBy) wins when the SAME expensive shuffle would otherwise repeat:
   - many JOINs on the same key, or
   - non-combinable aggregations (collect_list, percentile) over the full data.
   Then paying one shuffle up front and reusing the layout beats re-shuffling every time.
""")

pre_partitioned.unpersist()
spark.conf.set("spark.sql.shuffle.partitions", "200")

Shuffle every time:      0.469s
Pre-partition + cache:   0.722s

⚠️  Here pre-partitioning is usually NOT faster — and that itself is the lesson.
   These aggregations use combinable functions (avg / max / count), so each groupBy already
   does map-side partial aggregation (look for 'partial_avg' in the plan) and shuffles only a
   handful of rows per partition. The one-time repartition + cache adds a full shuffle that
   those already-cheap aggregations never get to amortize.

   Pre-partitioning (or bucketBy) wins when the SAME expensive shuffle would otherwise repeat:
   - many JOINs on the same key, or
   - non-combinable aggregations (collect_list, percentile) over the full data.
   Then paying one shuffle up front and reusing the layout beats re-shuffling every time.



### 5.2 Eliminating Shuffle via Broadcast Join (Review)

In [12]:
# Large table + small table join
dept_info = spark.createDataFrame([
    ("Engineering", "Tech Division", "CTO"),
    ("Marketing", "Marketing Division", "CMO"),
    ("Sales", "Sales Division", "CSO"),
    ("HR", "HR Division", "CHO"),
    ("Finance", "Finance Division", "CFO")
], ["department", "division_name", "executive"])

# Shuffle Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
start = time.time()
df.join(dept_info, "department").count()
shuffle_time = time.time() - start

# Broadcast Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
start = time.time()
df.join(F.broadcast(dept_info), "department").count()
broadcast_time = time.time() - start

print(f"Shuffle Join:    {shuffle_time:.3f}s")
print(f"Broadcast Join:  {broadcast_time:.3f}s")
print(f"\n💡 Always use broadcast() for small tables → Shuffle completely eliminated")

Shuffle Join:    0.686s
Broadcast Join:  0.219s

💡 Always use broadcast() for small tables → Shuffle completely eliminated


### 5.3 reduceByKey vs groupByKey (RDD API)

In [13]:
# Shuffle optimization at RDD level
rdd = sc.parallelize([
    (dept, random.randint(50000, 150000))
    for dept in random.choices(departments, k=1_000_000)
])

# ❌ groupByKey: Shuffle all data first, then group
start = time.time()
result1 = rdd.groupByKey().mapValues(lambda vals: sum(vals) / len(list(vals))).collect()
gbk_time = time.time() - start

# ✅ reduceByKey: local Combine first, then Shuffle
start = time.time()
result2 = rdd.mapValues(lambda v: (v, 1)) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda v: v[0] / v[1]) \
    .collect()
rbk_time = time.time() - start

print(f"groupByKey:  {gbk_time:.3f}s")
print(f"reduceByKey: {rbk_time:.3f}s")

print(f"""
💡 groupByKey vs reduceByKey

groupByKey:                  reduceByKey:
┌─────┐   Shuffle   ┌─────┐  ┌─────┐  Combine  ┌─────┐  Shuffle  ┌─────┐
│(A,1)│────────────→│(A,1)│  │(A,1)│──────────→│(A,4)│─────────→│(A,4)│
│(A,3)│────────────→│(A,3)│  │(A,3)│           └─────┘          │(A,9)│
│(B,2)│ move all    │(A,5)│  │(B,2)│                            └─────┘
└─────┘             └─────┘  └─────┘  local aggregate first,
                                       then Shuffle only small data
""")

groupByKey:  0.308s
reduceByKey: 0.249s

💡 groupByKey vs reduceByKey

groupByKey:                  reduceByKey:
┌─────┐   Shuffle   ┌─────┐  ┌─────┐  Combine  ┌─────┐  Shuffle  ┌─────┐
│(A,1)│────────────→│(A,1)│  │(A,1)│──────────→│(A,4)│─────────→│(A,4)│
│(A,3)│────────────→│(A,3)│  │(A,3)│           └─────┘          │(A,9)│
│(B,2)│ move all    │(A,5)│  │(B,2)│                            └─────┘
└─────┘             └─────┘  └─────┘  local aggregate first,
                                       then Shuffle only small data



---
## 6. Data Skew Problem and Solutions

When data concentrates on a specific key, only one partition becomes slow, stalling the entire job.

In [14]:
# Generate skewed data: 90% concentrated in "Engineering"
skew_data = []
for i in range(500_000):
    if random.random() < 0.9:
        dept = "Engineering"
    else:
        dept = random.choice(["Marketing", "Sales", "HR", "Finance"])
    skew_data.append((i, dept, random.randint(50000, 150000)))

skew_df = spark.createDataFrame(skew_data, ["id", "department", "salary"])

# Check skew distribution
skew_df.groupBy("department").count().orderBy(F.col("count").desc()).show()
print("⚠️ Data is extremely skewed toward Engineering.")

+-----------+------+
| department| count|
+-----------+------+
|Engineering|449899|
|    Finance| 12579|
|         HR| 12578|
|  Marketing| 12533|
|      Sales| 12411|
+-----------+------+

⚠️ Data is extremely skewed toward Engineering.


In [15]:
# groupBy under skewed conditions
spark.conf.set("spark.sql.shuffle.partitions", "5")

start = time.time()
skew_df.groupBy("department").agg(
    F.count("*").alias("cnt"),
    F.avg("salary").alias("avg_salary")
).collect()
skew_time = time.time() - start

print(f"Skewed groupBy: {skew_time:.3f}s")
print("(Note: this stays fast despite the skew — count/avg are combinable, so the hot key is")
print(" partially aggregated map-side before the shuffle. groupBy(avg) is NOT skew-sensitive.)")

# But the underlying partition distribution IS heavily skewed. This imbalance is what hurts
# operations that cannot combine map-side: shuffled JOINs and collect_list / percentile / etc.
show_partition_info(
    skew_df.repartition(5, "department"),
    "repartition by department (skewed) — one key dominates a single partition"
)

Skewed groupBy: 0.162s
(Note: this stays fast despite the skew — count/avg are combinable, so the hot key is
 partially aggregated map-side before the shuffle. groupBy(avg) is NOT skew-sensitive.)

repartition by department (skewed) — one key dominates a single partition
Partitions: 2
Total rows: 500,000
Rows/partition — min: 25,111, max: 474,889, avg: 250,000
Skew ratio (max/min): 18.9x
⚠️  Severe data skew detected!


In [16]:
# Fix 1: Salting — append a random suffix to the hot key so its rows spread across partitions
num_salts = 32

salted = skew_df.withColumn(
    "salted_key",
    F.concat(F.col("department"), F.lit("_"), (F.rand() * num_salts).cast("int").cast("string"))
)

# --- Deterministic evidence: size of the HOTTEST reduce partition (the straggler task) ---
# The right metric for skew is the MAX partition, not the min/max ratio: salting shrinks the
# single fat partition (small keys also split into tiny buckets, so min stays low — ignore it).
def hottest_partition(df, n, key):
    counts = [r["c"] for r in (df.repartition(n, key)
              .withColumn("p", F.spark_partition_id())
              .groupBy("p").agg(F.count("*").alias("c")).collect())]
    return max(counts)

before_max = hottest_partition(skew_df, num_salts, "department")     # hot key floods one partition
after_max  = hottest_partition(salted,  num_salts, "salted_key")     # hot key split into num_salts buckets
print(f"Hottest partition BEFORE salting (by 'department') : {before_max:,} rows")
print(f"Hottest partition AFTER  salting (by 'salted_key') : {after_max:,} rows")
print(f"  -> the straggler task shrank ~{before_max/after_max:.0f}x\n")

# --- Two-pass aggregation: aggregate on the salted key, then fold back to the real key ---
start = time.time()
partial = salted.groupBy("salted_key", "department").agg(
    F.count("*").alias("cnt"),
    F.sum("salary").alias("sum_salary")
)
result = partial.groupBy("department").agg(
    F.sum("cnt").alias("cnt"),
    (F.sum("sum_salary") / F.sum("cnt")).alias("avg_salary")
).collect()
salt_time = time.time() - start

print(f"plain  groupBy(avg) : {skew_time:.3f}s")
print(f"salted groupBy(avg) : {salt_time:.3f}s")
print("""
⚠️  For count/avg/sum, salting does NOT speed this up — it is usually SLOWER, because the
   plain groupBy already combines the hot key map-side (so skew never reaches the reduce
   side), while salting just adds a second aggregation pass.

   The hottest-partition numbers above are the real lesson: hashing on 'department' dumps the
   450K-row hot key into ONE partition; salting splits it into num_salts buckets, so the
   biggest reduce task shrinks ~10x. That matters precisely when the full hot partition
   reaches a single reduce task:
     - skewed shuffle JOINs (one task receives all matching rows for the hot key), and
     - non-combinable aggregations (collect_list, percentile, count(distinct)).
   At real scale that one fat partition is a multi-minute straggler; here it is too small to
   feel, but the imbalance is identical. (AQE skewJoin, shown next, automates this for joins.)
""")

spark.conf.set("spark.sql.shuffle.partitions", "200")

Hottest partition BEFORE salting (by 'department') : 449,899 rows
Hottest partition AFTER  salting (by 'salted_key') : 44,183 rows
  -> the straggler task shrank ~10x

plain  groupBy(avg) : 0.162s
salted groupBy(avg) : 0.277s

⚠️  For count/avg/sum, salting does NOT speed this up — it is usually SLOWER, because the
   plain groupBy already combines the hot key map-side (so skew never reaches the reduce
   side), while salting just adds a second aggregation pass.

   The hottest-partition numbers above are the real lesson: hashing on 'department' dumps the
   450K-row hot key into ONE partition; salting splits it into num_salts buckets, so the
   biggest reduce task shrinks ~10x. That matters precisely when the full hot partition
   reaches a single reduce task:
     - skewed shuffle JOINs (one task receives all matching rows for the hot key), and
     - non-combinable aggregations (collect_list, percentile, count(distinct)).
   At real scale that one fat partition is a multi-minute str

In [17]:
# Fix 2: AQE (Adaptive Query Execution) — Spark 3.x
# At runtime, automatically coalesces partitions or handles skew

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("AQE related settings:")
for key in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
]:
    try:
        print(f"  {key} = {spark.conf.get(key)}")
    except:
        print(f"  {key} = (not set)")

print(f"""
💡 AQE (Adaptive Query Execution):
   - After Shuffle, looks at actual data stats and auto-adjusts partitions
   - Merges small partitions (coalesce)
   - Splits skewed partitions (skew join optimization)
   - Changes Join strategy at runtime (SortMerge → Broadcast)
   
   Recommended to enable by default in Spark 3.x.
""")

# Disable for manual observation during lab
spark.conf.set("spark.sql.adaptive.enabled", "false")

AQE related settings:
  spark.sql.adaptive.enabled = true
  spark.sql.adaptive.skewJoin.enabled = true
  spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5.0
  spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 268435456b
  spark.sql.adaptive.coalescePartitions.enabled = true
  spark.sql.adaptive.coalescePartitions.minPartitionSize = 1048576b

💡 AQE (Adaptive Query Execution):
   - After Shuffle, looks at actual data stats and auto-adjusts partitions
   - Merges small partitions (coalesce)
   - Splits skewed partitions (skew join optimization)
   - Changes Join strategy at runtime (SortMerge → Broadcast)
   
   Recommended to enable by default in Spark 3.x.



---
## 7. Monitoring Shuffle in the Spark UI

What you can check at http://localhost:4040:

In [18]:
# Run a Shuffle-triggering operation and inspect it in the Spark UI
spark.conf.set("spark.sql.shuffle.partitions", "20")

result = (
    df.groupBy("department")
    .agg(
        F.count("*").alias("count"),
        F.avg("salary").alias("avg_salary"),
        F.stddev("salary").alias("stddev_salary")
    )
    .orderBy(F.col("avg_salary").desc())
)

result.show()

print("""
🔍 What to check in Spark UI (http://localhost:4040):

1. Jobs tab
   - How many Jobs were created
   - Number of Stages per Job

2. Stages tab
   - Shuffle Read / Write size
   - Number of Tasks per Stage
   - Task duration distribution (check for skew)

3. SQL tab
   - DAG visualization (Exchange node = Shuffle)
   - Row count and size per node

4. Stage Detail → Event Timeline
   - Execution time per task
   - Shuffle Read/Write time
   - GC time
""")

spark.conf.set("spark.sql.shuffle.partitions", "200")

+-----------+------+------------------+------------------+
| department| count|        avg_salary|     stddev_salary|
+-----------+------+------------------+------------------+
|      Sales|200733|100079.14426626413|28849.655445768472|
|    Finance|200009| 99998.82848271828|  28901.6090032703|
|Engineering|200620| 99990.12288904397| 28855.29907889875|
|  Marketing|199228| 99983.84617624029|28859.675260734166|
|         HR|199410| 99936.40588736773| 28858.37960142473|
+-----------+------+------------------+------------------+


🔍 What to check in Spark UI (http://localhost:4040):

1. Jobs tab
   - How many Jobs were created
   - Number of Stages per Job

2. Stages tab
   - Shuffle Read / Write size
   - Number of Tasks per Stage
   - Task duration distribution (check for skew)

3. SQL tab
   - DAG visualization (Exchange node = Shuffle)
   - Row count and size per node

4. Stage Detail → Event Timeline
   - Execution time per task
   - Shuffle Read/Write time
   - GC time



---
## 8. Partitioning Strategy for File Output

In [19]:
import shutil
import os

output_base = "/home/jovyan/data/output"

# Clean up previous run artifacts
for old_dir in ["/home/jovyan/work/spark-warehouse", f"{output_base}"]:
    if os.path.exists(old_dir):
        shutil.rmtree(old_dir)

# Method 1: Default output (one file per partition)
df.limit(100000).write.mode("overwrite").parquet(f"{output_base}/default")

# Method 2: Reduce file count with coalesce
df.limit(100000).coalesce(4).write.mode("overwrite").parquet(f"{output_base}/coalesced")

# Method 3: Directory partitioning with partitionBy (Hive style)
df.limit(100000).write.mode("overwrite").partitionBy("department").parquet(f"{output_base}/partitioned")

# Method 4: bucketBy (requires table registration)
spark.sql("DROP TABLE IF EXISTS bucketed_employees")
df.limit(100000).write.mode("overwrite") \
    .bucketBy(8, "department") \
    .sortBy("salary") \
    .saveAsTable("bucketed_employees")

print("File output complete! Checking the results of each method...")

File output complete! Checking the results of each method...


In [20]:
# Inspect the output file structure
for path_name in ["default", "coalesced", "partitioned"]:
    full_path = f"{output_base}/{path_name}"
    files = [f for f in os.listdir(full_path) if f.endswith(".parquet")]
    print(f"\n{path_name}: {len(files)} parquet file(s)")
    if path_name == "partitioned":
        dirs = [d for d in os.listdir(full_path) if d.startswith("department=")]
        print(f"  Partition directories: {dirs}")
        for d in sorted(dirs):
            sub_files = [f for f in os.listdir(f"{full_path}/{d}") if f.endswith(".parquet")]
            print(f"    {d}: {len(sub_files)} file(s)")


default: 1 parquet file(s)

coalesced: 1 parquet file(s)

partitioned: 0 parquet file(s)
  Partition directories: ['department=Finance', 'department=Engineering', 'department=HR', 'department=Sales', 'department=Marketing']
    department=Engineering: 1 file(s)
    department=Finance: 1 file(s)
    department=HR: 1 file(s)
    department=Marketing: 1 file(s)
    department=Sales: 1 file(s)


In [21]:
# Effect of partitionBy: read only specific partitions (Partition Pruning)
print("=== Full Scan ===")
all_scan = spark.read.parquet(f"{output_base}/partitioned")
all_scan.filter(F.col("department") == "Engineering").explain()

print("\n💡 When PartitionFilters shows department=Engineering,")
print("   only that directory is read (Partition Pruning).")
print("   Other partition directories are skipped entirely → massive I/O savings")

=== Full Scan ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#4008L,name#4009,salary#4010L,department#4011] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/output/partitioned], PartitionFilters: [isnotnull(department#4011), (department#4011 = Engineering)], PushedFilters: [], ReadSchema: struct<id:bigint,name:string,salary:bigint>



💡 When PartitionFilters shows department=Engineering,
   only that directory is read (Partition Pruning).
   Other partition directories are skipped entirely → massive I/O savings


---
## 📝 Key Summary

| Concept | Description |
|------|------|
| **Shuffle** | Redistribution of data between nodes by key. Causes network + disk I/O |
| **Narrow vs Wide** | map/filter have no Shuffle; groupBy/join/sort cause Shuffle |
| **repartition** | Full Shuffle, even distribution, can increase or decrease partitions |
| **coalesce** | No Shuffle, can only decrease partitions, may be uneven |
| **Data Skew** | Data concentrated on a specific key → fix with Salting or AQE |
| **Map-side combine** | count/avg/sum aggregate partially before the shuffle → these groupBys are NOT skew-sensitive |
| **Broadcast Join** | Broadcast small table → Shuffle completely eliminated |
| **partitionBy** | Hive-style directory partitioning → Partition Pruning |
| **bucketBy** | Hash-based bucketing → eliminates Shuffle during joins |
| **AQE** | Auto-adjusts partitions at runtime (Spark 3.x) |

### Shuffle Minimization Checklist
1. ✅ Use broadcast() for small tables
2. ✅ Use reduceByKey instead of groupByKey (RDD)
3. ✅ Pre-repartition + cache when an expensive shuffle is reused (joins / non-combinable aggs)
4. ✅ Apply Salting (or AQE skewJoin) for skewed JOINs and non-combinable aggregations
5. ✅ Tune spark.sql.shuffle.partitions
6. ✅ Enable AQE (Spark 3.x)
7. ✅ Use partitionBy / bucketBy for file output

> ⚠️ Skew note: `groupBy` with combinable aggregates (count/avg/sum) does map-side partial
> aggregation, so the hot key never floods a reduce task — those queries are not skew-sensitive
> and salting only adds overhead. Skew bites shuffled JOINs and non-combinable aggregations.

### Next Step (Step 4)
- Deep dive into Join strategies
- Broadcast Hash Join / Sort-Merge Join / Shuffle Hash Join
- Skew Join optimization
- Join order optimization

In [22]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
